# 🎵 Audio Classifier — Google Colab

**Workflow complet :**
1. Clone le repo GitHub
2. Installe les dépendances
3. Vérifie le GPU
4. Lance l'entraînement
5. Sauvegarde le modèle sur Google Drive

> ⚠️ **Avant de lancer** : `Exécution → Modifier le type d'exécution → GPU T4`

## Cellule 1 — Clone du repo + installation

In [ ]:
# ── Clone du repo ─────────────────────────────────────────────
# Remplace par l'URL de TON repo
REPO_URL = 'https://github.com/mat-bau/LELEC210X-project.git'
REPO_DIR = 'LELEC210X-project'   # nom du dossier local

import os

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print('✅ Repo cloné')
else:
    # Si tu relances la cellule, on pull les dernières modifs
    !cd {REPO_DIR} && git pull
    print('✅ Repo mis à jour')

# Se placer dans le dossier du projet
os.chdir(REPO_DIR)
print(f'📁 Répertoire courant : {os.getcwd()}')
!ls

In [ ]:
# ── Installation des dépendances depuis pyproject.toml ────────
# Colab utilise pip, pas uv — on extrait les dépendances automatiquement
import subprocess, sys

# Option A : si tu as aussi un requirements.txt généré par uv
# !pip install -r requirements.txt -q

# Option B : installation directe du package en mode editable
# (lit pyproject.toml et installe tout)
!pip install -e . -q

# Packages qui pourraient manquer sur Colab
!pip install librosa soundfile -q

print('✅ Dépendances installées')

## Cellule 2 — Vérification GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow : {tf.__version__}')

if gpus:
    print(f'✅ GPU détecté : {gpus[0]}')
    # Affiche le nom du GPU (T4, A100, etc.)
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else:
    print('❌ Pas de GPU — va dans Exécution → Modifier le type d\'exécution → GPU T4')
    raise RuntimeError('GPU requis pour un entraînement raisonnable')

## Cellule 3 — Montage Google Drive (sauvegarde du modèle)

In [ ]:
# Monte ton Google Drive pour sauvegarder le modèle entraîné.
# Colab redémarre après ~12h d'inactivité — sans ça tu perdrais tout.
from google.colab import drive

drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/audio_classifier/models/'
import os
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print(f'✅ Drive monté. Modèles sauvegardés dans : {DRIVE_MODEL_DIR}')

## Cellule 4 — Config adaptée Colab

In [ ]:
import sys
sys.path.insert(0, '.')  # S'assure que le package local est trouvable

# ── Imports depuis ton projet ──────────────────────────────────
# (identique à ton notebook local, le sys.path ci-dessus règle les imports)
import os, pickle, numpy as np
import matplotlib.pyplot as plt
import librosa, librosa.effects
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from collections import Counter
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.utils import to_categorical

from classification.datasets import Dataset
from classification.utils.audio_student import AudioUtil, Feature_vector_DS

print('✅ Imports OK')

# ── Config Colab (optimisée GPU) ───────────────────────────────
# DIFFÉRENCES vs local :
#   BATCH_SIZE   : 32 → 256  (GPU T4 a 15 GB VRAM, on en profite)
#   MODEL_DIR    : redirigé vers Google Drive (persistant)
#   EPOCHS       : réduit car on converge plus vite avec gros batch

class Config:
    # ── Audio ──────────────────────────────────────────────────
    SAMPLE_RATE  = 11025
    N_MEL        = 64
    N_FFT        = 512
    HOP_LENGTH   = 128
    DURATION_MS  = 1000

    # ── Augmentation ───────────────────────────────────────────
    AUG_TIME_SHIFT   = True
    AUG_PITCH_SHIFT  = True
    AUG_TIME_STRETCH = True
    AUG_NOISE        = True
    AUG_SPEC_MASKING = True
    PITCH_SHIFT_RANGE  = (-3, 3)
    TIME_STRETCH_RANGE = (0.85, 1.15)
    NOISE_SIGMA        = 0.04

    # ── Architecture ───────────────────────────────────────────
    DROPOUT_RATE    = 0.5
    L2_REG          = 1e-4
    LABEL_SMOOTHING = 0.1
    MIXUP_ALPHA     = 0.3

    # ── Training — valeurs Colab GPU ───────────────────────────
    BATCH_SIZE    = 256    # ← 32 sur CPU, 256 sur GPU T4
    EPOCHS        = 150    # early stopping gère le reste
    LEARNING_RATE = 5e-4

    # ── Callbacks ──────────────────────────────────────────────
    EARLY_STOPPING_PATIENCE = 30   # plus agressif qu'en local

    # ── Chemin modèle → Google Drive ───────────────────────────
    MODEL_DIR = '/content/drive/MyDrive/audio_classifier/models/'

    # ── TTA ────────────────────────────────────────────────────
    TTA_STEPS = 5

    # ── Données réelles ────────────────────────────────────────
    # Chemin dans le repo cloné — à adapter si ta structure diffère
    TEST_ON_ACQUIRED_DATA = True
    USE_REAL_VAL          = True
    USE_EXTRA_TRAIN_DATA  = True

    # Ces chemins sont relatifs à la racine du repo (REPO_DIR)
    ACQUIRED_DATA    = 'mcu/hands_on_audio_acquisition/audio_files'
    EXTRA_TRAIN_DIR  = 'mcu/hands_on_audio_acquisition/audio_files'
    EXTRA_AUG_PASSES = 40

    # MCU
    MCU_SAMPLE_RATE = 11025

config = Config()
print('✅ Config Colab chargée')
print(f'   BATCH_SIZE={config.BATCH_SIZE} | EPOCHS={config.EPOCHS} | LR={config.LEARNING_RATE}')
print(f'   Modèle sauvegardé dans : {config.MODEL_DIR}')

## Cellule 5 — Entraînement

In [ ]:
# Colle ici les fonctions de ton classifier_resnet.py
# (FeatureExtractor, prepare_dataset, build_resnet_audio, train_model, etc.)
# OU importe-les directement si ton fichier est dans le repo :

# Option 1 — import direct (si classifier_resnet.py est dans classification/)
# from classification.classifier_resnet import (
#     FeatureExtractor, prepare_dataset, build_resnet_audio,
#     train_model, evaluate_full, main_pipeline
# )

# Option 2 — colle le code ici (si le fichier n'est pas encore un module)
# %load classification/classifier_resnet.py

# ── Lance le pipeline ──────────────────────────────────────────
dataset = Dataset()
dataset.remove_class('background')
dataset.remove_class('birds')
dataset.remove_class('handsaw')
dataset.remove_class('helicopter')

model, metrics, classnames, extractor = main_pipeline(dataset, config)

## Cellule 6 — Récupération du modèle entraîné

In [ ]:
# Le modèle est déjà sauvegardé sur Drive automatiquement.
# Cette cellule te montre comment le télécharger sur ton PC si besoin.

from google.colab import files
import shutil, os

# Crée un zip du dossier modèle pour téléchargement facile
DRIVE_MODEL_DIR = '/content/drive/MyDrive/audio_classifier/models/'
zip_path = '/content/models_trained.zip'

shutil.make_archive('/content/models_trained', 'zip', DRIVE_MODEL_DIR)
print(f'✅ Archive créée : {zip_path}')
print(f'   Contenu :')
!unzip -l {zip_path}

# Télécharge sur ton PC
files.download(zip_path)
print('⬇️  Téléchargement lancé')

## Cellule 7 — Utilitaires
Cellules pratiques pour le debug et la gestion de session.

In [ ]:
# ── Vérification RAM / VRAM disponible ────────────────────────
!nvidia-smi
from psutil import virtual_memory
ram = virtual_memory()
print(f'RAM : {ram.available/1e9:.1f} GB disponible / {ram.total/1e9:.1f} GB total')

In [ ]:
# ── Pull les dernières modifs du repo sans tout refaire ────────
# Lance cette cellule si tu as poussé des changements depuis ton PC
!git pull
print('✅ Repo mis à jour — recharge les imports si tu as modifié du code Python')

In [ ]:
# ── Recharger un module Python sans redémarrer le kernel ───────
# Utile si tu modifies classifier_resnet.py et veux retester
import importlib
# import classification.classifier_resnet as clf
# importlib.reload(clf)
# from classification.classifier_resnet import main_pipeline
print('Décommente les lignes ci-dessus avec ton module')